Train a credit approval model using XGBoost that explicitly minimizes expected economic cost arising from asymmetric decision errors.

cost_fp = 100   # approving a bad borrower

cost_fn = 10    # rejecting a good borrower

In [27]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

from xgboost import XGBClassifier

In [28]:
cost_fp = 100   # approving a bad borrower
cost_fn = 10    # rejecting a good borrower

In [29]:
df = pd.read_csv("../data/credit_dataset.csv")

X = df[[
    "age",
    "income",
    "debt_to_income",
    "credit_score",
    "loan_amount"
]]

y = df["approved"]

FileNotFoundError: [Errno 2] No such file or directory: '../data/credit_dataset.csv'

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [22]:
sample_weights = np.where(
    y_train == 0,
    cost_fp,   # bad borrower approved → expensive
    cost_fn    # good borrower rejected → cheaper
)

In [23]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(
    X_train,
    y_train,
    sample_weight=sample_weights
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [24]:
y_pred = xgb_model.predict(X_test)

In [25]:
cm = confusion_matrix(y_test, y_pred)

TN, FP, FN, TP = cm.ravel()

expected_cost = FP * cost_fp + FN * cost_fn
cost_per_applicant = expected_cost / len(y_test)

cm, expected_cost, cost_per_applicant

(array([[1160,   40],
        [ 434, 1366]]),
 8340,
 2.78)

Why FP dropped or increased
FP increased because the model 

Why FN moved in the opposite direction

Why total cost improved or worsened